# A-5b M 통합 실모델 확인 (개발 세트)

Colab **T4 GPU** 런타임에서 위에서부터 순서대로 실행한다.

- **개발 세트(`data/frozen/dev.jsonl`)만** 쓴다. 평가 세트로 프롬프트를 고치면 감점이다.
- M 파이프라인(`build_m`)은 `scripts.run_eval`이 만든다. 모델 태그·상한·예산·판정 형식(`judge.format`)은 모두 `configs/default.yaml`에서 읽는다.
- 로그는 문항마다 즉시 `results/runs/dev/r2/m.jsonl`에 쓴다. 셀이 끊겨도 같은 셀을 다시 실행하면 이어 돈다.
- 라운드는 **`r2`**를 쓴다. `r1`에는 B-4의 P1 개발 로그가 있고, 그 `RUN_META.json`의 설정 스냅샷에는 `judge` 섹션이 없어 이어 돌 수 없다.
- 실행 전에 로컬의 `YongMin` 브랜치가 GitHub에 push되어 있어야 한다.

| 볼 지표 | 기대 |
|---|---|
| 완주·오류 | 60문항 모두 기록, `error`·`timeout` 0 |
| 호출 수 | 문항당 4 이하, `budget_exhausted` 0 |
| 판정 잘림·파싱 실패 | 0에 가깝게 (C-4: 0%) |
| 정확도·호출 수·시간 | P1(`results/runs/dev/r1/p1.jsonl`)과 비교 |
| 기권률 | 답 있는 문항에서 과잉 기권이 크지 않은가 (C-4가 지적한 문제) |


## 0. GPU 확인

In [1]:
!nvidia-smi --query-gpu=name,memory.used,memory.total --format=csv

name, memory.used [MiB], memory.total [MiB]
Tesla T4, 0 MiB, 15360 MiB


## 1. 코드 받기 (`YongMin` 브랜치)

In [2]:
import os
os.chdir("/content")   # 다시 실행해도 repo/repo가 생기지 않게 항상 같은 위치에서 시작
REPO_URL = "https://github.com/minyi-k03/Intelligence-Software-Project.git"
# 비공개 저장소면 Colab 비밀(userdata)에 GH_TOKEN을 넣고 아래 두 줄 주석을 푼다. 토큰을 노트북에 적지 말 것.
# from google.colab import userdata
# REPO_URL = REPO_URL.replace("https://", f"https://{userdata.get('GH_TOKEN')}@")

if not os.path.isdir("repo"):
    !git clone {REPO_URL} repo
%cd /content/repo
!git fetch origin
!git checkout YongMin
!git pull --ff-only origin YongMin
!git log --oneline -3
assert os.path.isfile("tests/test_m_dev_smoke.py"), "A-5b 커밋이 없다. YongMin 브랜치가 push됐는지 확인할 것"


Cloning into 'repo'...
remote: Enumerating objects: 642, done.
remote: Counting objects: 100% (287/287), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 642 (delta 189), reused 200 (delta 132), pack-reused 355 (from 1)
Receiving objects: 100% (642/642), 658.32 KiB | 14.01 MiB/s, done.
Resolving deltas: 100% (340/340), done.
/content/repo
branch 'YongMin' set up to track 'origin/YongMin'.
Switched to a new branch 'YongMin'
From https://github.com/minyi-k03/Intelligence-Software-Project
 * branch            YongMin    -> FETCH_HEAD
Already up to date.
69b824e (HEAD -> YongMin, origin/YongMin) fix(meba): A-5b 교차 검증 반영 - 노트북 요약 지표 규칙 정리와 판정 형식 종단 단언
1496f49 fix(meba): A-5b 리뷰 반영 - tag 경로 종단 스모크, 중단 후 이어 돌기, 형식 값 어긋남 방지
cf5a0a8 feat(meba): PR A-5b M 통합 - 판정 포맷 설정(judge.format)과 dev 전체 스모크


## 2. 의존성

In [3]:
!pip install -q -r requirements.txt && pip list 2>/dev/null | grep -iE "^(langchain|langgraph|langchain-ollama) "

langchain                             1.4.8
langchain-ollama                      1.2.0
langgraph                             1.2.19


## 3. Ollama 설치와 서버 실행

In [4]:
# lspci/lshw가 없으면 설치 스크립트가 GPU를 못 찾는다 (Colab 이미지에는 기본으로 없음) -> 먼저 설치
!apt-get install -y -q zstd pciutils lshw > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
!pkill -f "ollama serve" || true      # 이전 실행에서 남은 서버가 있으면 끈다
time.sleep(2)

LOG = "/content/ollama.log"           # 서버 로그를 버리지 않고 남긴다
server = subprocess.Popen(["ollama", "serve"], stdout=open(LOG, "w"), stderr=subprocess.STDOUT,
                          start_new_session=True)   # 셀을 중단(Ctrl-C)해도 서버가 같이 죽지 않게 한다
for _ in range(60):                   # 응답할 때까지 최대 60초 기다린다
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        print("서버 준비됨")
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(f"Ollama 서버가 60초 안에 뜨지 않았다. 서버 로그 끝: {open(LOG).read()[-2000:]}")
!ollama --version
print("--- 서버 로그에서 GPU 관련 줄")
!grep -iE "cuda|gpu|inference compute|library" {LOG} | head -10

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
^C
서버 준비됨
ollama version is 0.40.18
--- 서버 로그에서 GPU 관련 줄
time=2026-11-26T00:45:00.957Z level=INFO source=routes.go:2484 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GGML_VK_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: LLAMA_ARG_FIT: LLAMA_ARG_FIT_TARGET: NO_PROXY: OLLAMA_CONTEXT_LENGTH:0 OLLAMA_CREATE_REMOTE:false OLLAMA_DEBUG:INFO OLLAMA_DEBUG_LOG_REQUESTS:false OLLAMA_EDITOR: OLLAMA_FLASH_ATTENTION:false OLLAMA_GO_TEMPLATE:true OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0

## 4. 모델 받기 (태그는 `configs/default.yaml`에서 읽는다)

In [5]:
import subprocess, yaml
tag = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print("model.tag =", tag)
r = subprocess.run(["ollama", "pull", tag], capture_output=True, text=True)   # 진행 막대는 버리고 결과만 본다
print("pull 종료 코드:", r.returncode, "|", r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)

model.tag = qwen2.5:7b-instruct-q4_K_M
pull 종료 코드: 0 | 25hpulling manifest 
pulling 2bada8a74506: 100% ▕██████████████████▏ 4.7 GB                         
pulling 2f15b3218f05: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 

NAME                          ID              SIZE      MODIFIED               
qwen2.5:7b-instruct-q4_K_M    845dbda0ea48    4.7 GB    Less than a second ago    



## 4-1. 모델이 GPU로 올라갔는지 확인
`PROCESSOR`가 `100% GPU`여야 한다. `CPU`가 보이면 멈추고 확인한다 (지연 시간이 크게 늘어난다).

In [6]:
import subprocess, yaml
tag = yaml.safe_load(open("configs/default.yaml", encoding="utf-8"))["model"]["tag"]
print(subprocess.run(["ollama", "run", tag, "Say OK."], capture_output=True, text=True, timeout=300).stdout)
ps = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout
print(ps)
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
assert "GPU" in ps and "CPU" not in ps.replace("PROCESSOR", ""), "모델이 GPU로 올라가지 않았다. 위 서버 로그와 ollama ps를 확인할 것"
print("OK: GPU로 로드됨")

OK.


NAME                          ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen2.5:7b-instruct-q4_K_M    845dbda0ea48    4.8 GB    100% GPU     4096       llamacpp    4 minutes from now    

memory.used [MiB], memory.total [MiB]
4663 MiB, 15360 MiB
OK: GPU로 로드됨


## 5. 단위 테스트 (GPU 없이 도는 테스트)

가짜 모델 스모크(`test_m_dev_smoke.py`)가 실제 코퍼스·개발 세트 60문항으로 로그 규격을 이미 확인한다. 여기서는 환경이 같은지만 본다.

In [7]:
!python -m pytest tests/test_graph.py tests/test_judge.py tests/test_requery.py tests/test_config.py tests/test_m_dev_smoke.py -q 2>&1 | tail -3

........................................................................ [ 62%]
............................................                             [100%]
116 passed in 4.64s


## 6. 동결 해시 확인
해시가 `data/README.md` 표와 다르면 여기서 멈춘다 (`run_eval`도 시작 전에 다시 확인한다).

In [8]:
import json
from collections import Counter
from pathlib import Path

from src.eval.logger import verify_frozen_hashes

hashes = verify_frozen_hashes(Path("data/frozen"), Path("data/README.md"))
print({name: digest[:12] for name, digest in hashes.items()})
items = [json.loads(line) for line in Path("data/frozen/dev.jsonl").open(encoding="utf-8")]
print(len(items), "문항", Counter(item["type"] for item in items))
!grep -n "format" configs/default.yaml


{'eval.jsonl': '44de0d8a6cf7', 'dev.jsonl': '8f18f76dd903', 'corpus.jsonl': 'd562adc17741'}
60 문항 Counter({'hotpotqa': 20, 'squad2': 20, 'gsm8k': 20})
28:  format: "json"                    # Evidence-judgment output format: "json" (ADR-0004, confirmed in Experiment C-4) or "tag" (control)


## 7. 검증 모드 (앞 3문항, `_scratch` 묶음)

실모델이 한 문항이라도 끝까지 도는지, 로그가 규격대로 나오는지 먼저 본다. 이 결과는 git 제외 폴더에 쓴다.

In [9]:
!python -m scripts.run_eval --verify 3 --pipelines m
rows = [json.loads(l) for l in Path("results/runs/_scratch/r1/m.jsonl").open(encoding="utf-8")]
for r in rows:
    print(r["qid"], "|", r["route"], r["decisions"], "| 답:", repr(r["answer"]), "| 정답:", r["gold"], "| calls:", r["n_calls"], "| error:", r["error"])
print(sorted(rows[0]))


[m r1]   1 hotpotqa-0040 wrong 13.4s
[m r1]   2 hotpotqa-0041 ok 2.2s
[m r1]   3 hotpotqa-0042 wrong 4.0s
완료: _scratch/r1 최대 VRAM(MiB) {'m': 5061}
hotpotqa-0040 | search ['RETRIEVE', 'RETRIEVE', 'ABSTAIN'] | 답: 'UNANSWERABLE' | 정답: ['Aristotle'] | calls: 4 | error: None
hotpotqa-0041 | search ['ANSWER'] | 답: 'United States Auto Club' | 정답: ['United States Auto Club'] | calls: 2 | error: None
hotpotqa-0042 | search ['RETRIEVE', 'ANSWER'] | 답: 'Reinvestment risk' | 정답: ['terrorist activity'] | calls: 3 | error: None
['abstained', 'answer', 'answer_kind', 'budget_exhausted', 'bundle', 'calls', 'correct', 'decisions', 'em', 'error', 'evidence_kept', 'f1', 'gold', 'is_warmup', 'kept_windows', 'latency_s', 'loop_breaks', 'max_input_tokens', 'n_calls', 'n_retries', 'numeric_correct', 'order_idx', 'pipeline', 'qid', 'retrieved_doc_ids', 'round', 'route', 'session_idx', 'subtype', 'timeout', 'type']


## 8. 개발 세트 60문항 전체 실행

문항마다 즉시 저장한다. 끊기면 같은 셀을 다시 실행하면 이어 돈다. 설정을 바꿨다면 `r3`처럼 새 라운드로 돌려야 한다 (한 라운드에 조건이 섞이면 러너가 멈춘다).

In [10]:
!python -m scripts.run_eval --bundle dev --round r2 --pipelines m

[m r2]   1 hotpotqa-0040 wrong 7.4s
[m r2]   2 hotpotqa-0041 ok 2.2s
[m r2]   3 hotpotqa-0042 wrong 4.2s
[m r2]   4 hotpotqa-0043 wrong 8.0s
[m r2]   5 hotpotqa-0044 wrong 7.9s
[m r2]   6 hotpotqa-0045 wrong 9.2s
[m r2]   7 hotpotqa-0046 ok 4.9s
[m r2]   8 hotpotqa-0047 wrong 3.6s
[m r2]   9 hotpotqa-0048 wrong 9.2s
[m r2]  10 hotpotqa-0049 wrong 2.7s
[m r2]  11 hotpotqa-0050 wrong 4.6s
[m r2]  12 hotpotqa-0051 wrong 8.8s
[m r2]  13 hotpotqa-0052 ok 2.1s
[m r2]  14 hotpotqa-0053 wrong 7.8s
[m r2]  15 hotpotqa-0054 wrong 8.1s
[m r2]  16 hotpotqa-0055 wrong 2.9s
[m r2]  17 hotpotqa-0056 wrong 2.6s
[m r2]  18 hotpotqa-0057 wrong 2.4s
[m r2]  19 hotpotqa-0058 wrong 2.5s
[m r2]  20 hotpotqa-0059 wrong 2.8s
[m r2]  21 squad2-0040 wrong 7.8s
[m r2]  22 squad2-0041 wrong 2.1s
[m r2]  23 squad2-0042 wrong 2.4s
[m r2]  24 squad2-0043 ok 5.7s
[m r2]  25 squad2-0044 wrong 8.4s
[m r2]  26 squad2-0045 wrong 8.1s
[m r2]  27 squad2-0046 wrong 7.2s
[m r2]  28 squad2-0047 wrong 7.9s
[m r2]  29 squad2-00

## 9. 요약표

집계 규칙은 `EVALUATION §4`·`INTERFACES §7.2`를 따른다.

- **정확도·기권·잘림·파싱 지표**: 워밍업 문항을 **포함한 60문항 전체**
- **비용·시간 지표**(`n_calls`, `out_tokens`, `latency_s`): 파이프라인의 첫 1문항(워밍업)을 **제외한 59문항**

| 열 | 뜻 |
|---|---|
| `retries` | 잘림 재시도 횟수 합 |
| `judge_trunc_first` | 판정 호출의 **1차 응답**이 출력 상한에 걸린 문항 비율 (재시도로 복구된 것 포함) |
| `judge_trunc` | **재시도 뒤에도** 잘린 판정 호출이 있는 문항 비율 (`scripts/compare_judge_format.py`의 `truncated_after_retry`와 같은 기준) |
| `parse_fail` | 판정 파싱 실패가 있는 문항 비율 |
| `over_abstain` | 답이 있는 문항(`answer_kind != unanswerable`) 중 기권한 비율 |


In [11]:
import pandas as pd

OUT = Path("results/runs/dev/r2/m.jsonl")
df = pd.DataFrame([json.loads(l) for l in OUT.open(encoding="utf-8")])
df["group"] = df.apply(lambda r: f"{r['type']}-{r['subtype']}" if r["type"] == "squad2" else r["type"], axis=1)
judge_calls = df["calls"].map(lambda cs: [c for c in cs if c["step"] == "judge"])
df["judge_trunc_first"] = judge_calls.map(lambda cs: any(c["truncated"] and not c["is_retry"] for c in cs))
df["judge_trunc"] = judge_calls.map(lambda cs: any(c["truncated"] and c["is_retry"] for c in cs))
df["parse_fail"] = judge_calls.map(lambda cs: any(c["parse_ok"] is False for c in cs))
df["total_out"] = df["calls"].map(lambda cs: sum(c["output_tokens"] for c in cs))
df["over_abstain"] = df["abstained"] & (df["answer_kind"] != "unanswerable")
cost = df[~df["is_warmup"]]   # 비용·시간 집계만 워밍업 제외. 정확도·부작용 지표는 전체(df)

print("완주:", len(df), "/ 60 | error:", int(df["error"].notna().sum()), "| timeout:", int(df["timeout"].sum()),
      "| budget_exhausted:", int(df["budget_exhausted"].sum()))
quality = df.groupby("group").agg(n=("qid", "count"), correct=("correct", "mean"), abstained=("abstained", "mean"),
                                  retries=("n_retries", "sum"), judge_trunc_first=("judge_trunc_first", "mean"),
                                  judge_trunc=("judge_trunc", "mean"), parse_fail=("parse_fail", "mean"))
spend = cost.groupby("group").agg(n_calls=("n_calls", "mean"), out_tokens=("total_out", "mean"),
                                  latency_s=("latency_s", "mean"))
pd.set_option("display.width", 200)
print(quality.join(spend).round(3))
print(f"전체 {len(df)}문항 | 정확도:", round(df["correct"].mean(), 3), "| 과잉 기권:",
      round(df["over_abstain"].sum() / (df["answer_kind"] != "unanswerable").sum(), 3))
print(f"워밍업 제외 {len(cost)}문항 | 평균 호출:", round(cost["n_calls"].mean(), 2),
      "| 평균 출력 토큰:", round(cost["total_out"].mean(), 1), "| 평균 시간:", round(cost["latency_s"].mean(), 2), "s")
print("경로:", dict(Counter(df["route"])))
print("판정:", dict(Counter(d for ds in df["decisions"] for d in ds)))
print("evidence_kept(HotpotQA):", df["evidence_kept"].dropna().mean())


완주: 60 / 60 | error: 0 | timeout: 0 | budget_exhausted: 0
                      n  correct  abstained  retries  judge_trunc_first  judge_trunc  parse_fail  n_calls  out_tokens  latency_s
group                                                                                                                           
gsm8k                20     0.85       0.00        2                0.0          0.0         0.0    1.300     239.150      7.837
hotpotqa             20     0.15       0.65        0                0.0          0.0         0.0    2.895     129.368      5.081
squad2-answerable    10     0.20       0.60        0                0.0          0.0         0.0    3.100     134.200      5.399
squad2-unanswerable  10     0.70       0.70        0                0.0          0.0         0.0    2.500      94.700      3.565
전체 60문항 | 정확도: 0.483 | 과잉 기권: 0.38
워밍업 제외 59문항 | 평균 호출: 2.32 | 평균 출력 토큰: 161.5 | 평균 시간: 5.81 s
경로: {'search': 40, 'math': 20}
판정: {'RETRIEVE': 35, 'ABSTAIN': 26, 'ANSWE

## 10. P1과 비교 (같은 개발 세트, `r1`의 P1 로그)

정확도·정답 겹침은 60문항 전체, 호출 수·시간은 **양쪽 모두** 워밍업이 아닌 문항만 비교한다.


In [12]:
P1 = Path("results/runs/dev/r1/p1.jsonl")
p1 = pd.DataFrame([json.loads(l) for l in P1.open(encoding="utf-8")])
both = df.merge(p1[["qid", "correct", "n_calls", "latency_s", "is_warmup"]], on="qid", suffixes=("_m", "_p1"))
spent = both[~both["is_warmup_m"] & ~both["is_warmup_p1"]]   # 비용·시간 비교는 워밍업 제외
print(f"문항: {len(both)} (호출·시간 비교: {len(spent)}문항, 워밍업 제외)")
print("정확도  M:", round(both["correct_m"].mean(), 3), " P1:", round(both["correct_p1"].mean(), 3))
print("호출    M:", round(spent["n_calls_m"].mean(), 2), " P1:", round(spent["n_calls_p1"].mean(), 2))
print("시간    M:", round(spent["latency_s_m"].mean(), 2), " P1:", round(spent["latency_s_p1"].mean(), 2))
print("M만 정답:", int((both["correct_m"] & ~both["correct_p1"]).sum()), " P1만 정답:", int((~both["correct_m"] & both["correct_p1"]).sum()))


문항: 60 (호출·시간 비교: 59문항, 워밍업 제외)
정확도  M: 0.483  P1: 0.35
호출    M: 2.32  P1: 1.75
시간    M: 5.81  P1: 9.02
M만 정답: 14  P1만 정답: 6


## 11. 이상 문항 (원문 확인)

오류·시간 초과·판정 잘림(1차 포함)·파싱 실패·예산 소진 문항만 본다. 1차만 잘리고 재시도로 복구된 문항도 여기 나온다.


In [13]:
odd = df[df["error"].notna() | df["timeout"] | df["judge_trunc_first"] | df["judge_trunc"] | df["parse_fail"]
         | df["budget_exhausted"]]
print(f"{len(odd)}문항")
for _, r in odd.iterrows():
    print("=" * 100)
    print(f"{r['qid']} route={r['route']} decisions={r['decisions']} answer={r['answer']!r} error={r['error']}")
    print("  calls (step, out_tokens, truncated, is_retry, parse_ok):",
          [(c["step"], c["output_tokens"], c["truncated"], c["is_retry"], c["parse_ok"]) for c in r["calls"]])


0문항


## 12. 결과 내려받기

Colab 런타임이 끝나면 파일은 사라진다. 로그(`results/runs/dev/r2/`)를 내려받아 레포에 올리고, 해석은 `docs/experiments/`에 남긴다.

In [ ]:
!cd results/runs/dev && zip -qr /content/a5b_dev_r2.zip r2
from google.colab import files
files.download("/content/a5b_dev_r2.zip")


## 13. 판정 진단: 근거에 정답이 있었는데 RETRIEVE·ABSTAIN했나 (r2)

r2 결과에서 HotpotQA 기권 65%, SQuAD 답 있음 기권 60%, 과잉 기권 0.38이 나왔다. 그런데 `evidence_kept`(HotpotQA 근거 문장 보존율)는 0.9다. 근거는 모였는데 판정이 답을 확정하지 못한다는 뜻인지 확인한다.

- 로그의 `retrieved_doc_ids`·`kept_windows`로 판정 호출이 본 압축 근거를 회차별로 복원한다 (모델 호출 없음).
- "근거에 정답 있음" = 채점과 같은 정규화를 거친 근거 텍스트에 정답 문자열이 들어 있음 (C-4와 같은 기준). 정답 문자열이 있어도 그 문장만으로 답할 수 있다는 보장은 아니다.
- 대상: 검색 경로를 탄, 답이 있는 문항


In [ ]:
import json
from collections import Counter
from pathlib import Path

import pandas as pd

from src.config.settings import get_settings
from src.eval.scorer import _normalize          # 채점과 같은 정규화
from src.meba.compress import build_windows, render

ITEMS = {d["qid"]: d for d in map(json.loads, open("data/frozen/dev.jsonl", encoding="utf-8"))}
CORPUS = {d["doc_id"]: d for d in map(json.loads, open("data/frozen/corpus.jsonl", encoding="utf-8"))}
WINDOW = get_settings().compress.window


def load_round(rnd):
    """라운드의 M 로그를 DataFrame으로 읽는다."""
    rows = [json.loads(l) for l in Path(f"results/runs/dev/{rnd}/m.jsonl").open(encoding="utf-8")]
    df = pd.DataFrame(rows)
    df["group"] = df.apply(lambda r: f"{r['type']}-{r['subtype']}" if r["type"] == "squad2" else r["type"], axis=1)
    return df


def evidence_by_round(row):
    """판정 회차별 압축 근거 텍스트 (판정은 앞 회차 근거에 이어 붙인 것을 본다)."""
    out = []
    for doc_ids, kept in zip(row["retrieved_doc_ids"], row["kept_windows"]):
        paras = [CORPUS[i] for i in doc_ids]
        by_id = {w.window_id: w for w in build_windows(paras, WINDOW)}
        out.append(render(paras, [by_id[k] for k in kept if k in by_id]))
    return out


def diagnose(df):
    """검색 경로 · 답 있는 문항마다 결과와 '근거에 정답 있음'을 붙인 표."""
    recs = []
    for _, r in df[(df["route"] == "search") & (df["answer_kind"] != "unanswerable")].iterrows():
        golds = [_normalize(g) for g in ITEMS[r["qid"]]["answers"]]
        ev = [_normalize(e) for e in evidence_by_round(r)]
        outcome = "정답" if r["correct"] else ("기권" if r["abstained"] else "오답")
        recs.append({"qid": r["qid"], "group": r["group"], "outcome": outcome,
                     "decisions": "→".join(r["decisions"]), "n_calls": r["n_calls"],
                     "gold_in_first": bool(ev) and any(g and g in ev[0] for g in golds),
                     "gold_in_any": any(g and g in e for g in golds for e in ev)})
    return pd.DataFrame(recs)


def show_diagnosis(rnd):
    d = diagnose(load_round(rnd))
    print(f"[{rnd}] 검색 경로 · 답 있는 문항 {len(d)}개")
    print(pd.crosstab(d["outcome"], d["gold_in_any"].map({True: "근거에 정답 있음", False: "없음"}), margins=True))
    first_hit = d[d["gold_in_first"]]
    print(f"\n1차 근거에 이미 정답이 있던 {len(first_hit)}문항의 첫 판정:",
          dict(Counter(s.split("→")[0] for s in first_hit["decisions"])))
    return d


diag_r2 = show_diagnosis("r2")
print()
print(diag_r2[(diag_r2["outcome"] != "정답") & diag_r2["gold_in_any"]].to_string(index=False))


## 14. 판정 프롬프트 v2로 다시 실행 (r3)

13의 진단에서 "근거에 정답이 있는데 기권·재검색"이 많으면 판정 프롬프트를 고친다 (C-4 후속 작업과 같은 방향). v2에서 바꾼 것은 `src/meba/judge.py`의 지시문 세 군데다. **ABSTAIN 정의와 공통 답 형식 지시문(`CONTENT_INSTRUCTION`)은 그대로** 두었다.

| 자리 | v1 | v2 |
|---|---|---|
| ANSWER | the evidence states the answer. | 한 문장에 있거나 여러 문단을 합쳐 답을 댈 수 있으면, 질문과 표현이 달라도 ANSWER |
| RETRIEVE | 근거가 질문과 관련 있지만 필요한 사실 하나가 없음 | **아직 답을 댈 수 없고**, 필요한 사실 하나가 없음 |
| 빠진 정보 | 사람·장소·작품·용어를 적은 빠진 사실 | 같음 + 답 후보를 쓰지 말 것 |
| 서술(reasoning) | 근거가 답하는지 한두 문장 | 답이 되는 근거 부분을 **인용**하거나, 빠진 사실을 말함 |

- **반드시 새 라운드 `r3`로 돈다.** `RUN_META`는 설정 스냅샷만 비교하고 git 커밋은 기록만 하므로, 코드를 pull한 뒤 `r2`를 이어 돌리면 막히지 않고 v1·v2가 섞인다.
- 수학 경로(라우터)는 바뀌지 않았다. temperature 0이므로 GSM8K 결과가 r2와 같아야 한다 (15에서 확인).


In [ ]:
assert Path("results/runs/dev/r2/m.jsonl").exists(), "비교 기준 r2 로그가 없다 (런타임이 초기화됐으면 r2 zip을 먼저 올릴 것)"
!git pull --ff-only origin YongMin
!git log --oneline -1
!grep -n "quote the part of the evidence" src/meba/judge.py
!python -m pytest tests/test_judge.py tests/test_m_dev_smoke.py -q 2>&1 | tail -1
!python -m scripts.run_eval --bundle dev --round r3 --pipelines m


## 15. r2(v1) vs r3(v2) 비교

채택 기준은 미리 정한다. SPEC §6의 조건부 부품 기준을 그대로 쓴다: **유형별 정답 수가 줄지 않고 목표 지표가 좋아지면 채택.**

- 목표 지표: 과잉 기권률, 문항당 호출 수 (워밍업 제외)
- 따로 볼 것: SQuAD 답할 수 없음의 정답 수 (기권을 덜 하게 만들면 여기가 먼저 나빠진다)
- 문항별 정답→오답 변화는 목록으로 남긴다. McNemar 정확 검정은 60문항이라 참고용이다.


In [ ]:
import math


def mcnemar_exact(b, c):
    """양측 정확 McNemar p값 (불일치 쌍 b, c)."""
    n = b + c
    if n == 0:
        return 1.0
    return min(1.0, 2 * sum(math.comb(n, i) for i in range(min(b, c) + 1)) / 2 ** n)


def round_summary(df):
    cost = df[~df["is_warmup"]]
    q = df.groupby("group").agg(n=("qid", "count"), correct=("correct", "sum"), abstained=("abstained", "sum"))
    s = cost.groupby("group").agg(n_calls=("n_calls", "mean"), latency_s=("latency_s", "mean"))
    return q.join(s)


v1, v2 = load_round("r2"), load_round("r3")
print(pd.concat({"v1 (r2)": round_summary(v1), "v2 (r3)": round_summary(v2)}, axis=1).round(2).to_string())

answerable = lambda d: (d["answer_kind"] != "unanswerable").sum()
for name, d in (("v1 (r2)", v1), ("v2 (r3)", v2)):
    c = d[~d["is_warmup"]]
    print(f"{name}: 정답 {int(d['correct'].sum())}/60 | 과잉 기권 {(d['abstained'] & (d['answer_kind'] != 'unanswerable')).sum() / answerable(d):.2f}"
          f" | 호출 {c['n_calls'].mean():.2f} | 시간 {c['latency_s'].mean():.2f}s | 오류 {int(d['error'].notna().sum())}"
          f" | 예산 소진 {int(d['budget_exhausted'].sum())} | 판정 {dict(Counter(x for xs in d['decisions'] for x in xs))}")

both = v1.merge(v2[["qid", "correct", "answer", "decisions", "route"]], on="qid", suffixes=("_v1", "_v2"))
gained = both[~both["correct_v1"] & both["correct_v2"]]
lost = both[both["correct_v1"] & ~both["correct_v2"]]
print(f"\nv2만 정답 {len(gained)} / v1만 정답 {len(lost)} | McNemar 정확 p = {mcnemar_exact(len(gained), len(lost)):.3f}")
for title, part in (("v2에서 맞힌 문항", gained), ("v2에서 틀린 문항 (정답→오답)", lost)):
    print(f"\n--- {title}")
    for _, r in part.iterrows():
        print(f"{r['qid']:15s} {r['decisions_v1']} {r['answer_v1']!r} → {r['decisions_v2']} {r['answer_v2']!r} | 정답 {ITEMS[r['qid']]['answers']}")

math_rows = both[both["route_v1"] == "math"]
same_math = (math_rows["answer_v1"] == math_rows["answer_v2"]).all() and (math_rows["route_v2"] == "math").all()
print("\n수학 경로 결과가 r2와 같은가:", same_math)

# P1(r1)과 비교: 시간 초과 문항은 호출 기록이 없고 시간은 120초로 남는다 (EVALUATION §4) -> 함께 보고한다.
p1 = pd.DataFrame([json.loads(l) for l in Path("results/runs/dev/r1/p1.jsonl").open(encoding="utf-8")])
p1c = p1[~p1["is_warmup"]]
v2c = v2[~v2["is_warmup"]]
print(f"\nP1(r1): 정답 {int(p1['correct'].sum())}/60 | 호출 {p1c['n_calls'].mean():.2f} | 시간 평균 {p1c['latency_s'].mean():.2f}s"
      f" · 중앙값 {p1c['latency_s'].median():.2f}s | 시간 초과 {int(p1['timeout'].sum())}")
print(f"M v2(r3): 정답 {int(v2['correct'].sum())}/60 | 호출 {v2c['n_calls'].mean():.2f} | 시간 평균 {v2c['latency_s'].mean():.2f}s"
      f" · 중앙값 {v2c['latency_s'].median():.2f}s | 시간 초과 {int(v2['timeout'].sum())}")


In [ ]:
diag_r3 = show_diagnosis("r3")
print()
print(diag_r3[(diag_r3["outcome"] != "정답") & diag_r3["gold_in_any"]].to_string(index=False))


## 16. 결과 내려받기 (r2 · r3)

두 라운드 로그를 레포의 `results/runs/dev/`에 올리고, 해석은 `docs/experiments/2026-11-26-a-m-dev-judge-prompt.md`에 채운다.


In [ ]:
!cd results/runs/dev && zip -qr /content/a5b_dev_r2_r3.zip r2 r3
from google.colab import files
files.download("/content/a5b_dev_r2_r3.zip")
